<a href="https://colab.research.google.com/github/Kotyga/ML_toolkit/blob/main/BPE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ДЗ 1. Byte-level BPE-токенизатор

В этой работе необходимо реализовать и обучить Byte-level BPE-токенизатор на русскоязычном корпусе.

Основные этапы:

1. Построить базовый байтовый словарь и собрать статистику соседних пар.
2. Реализовать слияния и обучение словаря заданного размера.
3. Реализовать применение выученных слияний по их рангам, кодирование и декодирование текста.
4. Проверить восстановление исходного текста и изучить длины полученных последовательностей.

Заполните участки, отмеченные вызовом `todo()`. Обучение языковой модели в эту работу не входит. Для выполнения достаточно CPU; время обучения токенизатора зависит от объёма корпуса. Публикация токенизатора на Hugging Face — необязательный заключительный этап.

In [ ]:
# Установите зависимости в окружение текущего ядра.
%pip install --quiet datasets regex tqdm huggingface_hub

In [ ]:
import inspect
import json
import os
from collections import Counter
from functools import lru_cache
from pathlib import Path

import regex as re
from datasets import load_dataset
from huggingface_hub import HfApi, interpreter_login, snapshot_download
from tempfile import TemporaryDirectory
from tqdm.auto import tqdm, trange

SEED = 0xC0FFEE

In [ ]:
# Этой функцией будут помечены все места, которые необходимо дозаполнить
# Это могут быть как целые функции, так и отдельные части внутри них
# Всегда можно воспользоваться интроспекцией и найти места использования этой функции :)


def todo():
    stack = inspect.stack()
    caller_frame = stack[1]
    function_name = caller_frame.function
    line_number = caller_frame.lineno
    raise NotImplementedError(f"TODO at {function_name}, line {line_number}")

## Датасет

Используется корпус [IgorVolochay/russian_jokes](https://huggingface.co/datasets/IgorVolochay/russian_jokes). Загрузите его и изучите несколько примеров. Словарь обучается только на тренировочной части; отложенная часть используется для анализа токенизации.

In [ ]:
dataset = load_dataset("IgorVolochay/russian_jokes")
print("\n===\n".join(dataset["train"]["text"][:3]))

In [ ]:
# Подготовим холдауты
dataset = dataset["train"].train_test_split(test_size=0.1, seed=SEED)
dataset

## Реализация Byte-level BPE

1. Реализуйте `merge`: объединение выбранной пары и обновление статистики.
2. Реализуйте `train`: построение словаря и упорядоченного списка слияний.
3. Обучите токенизатор на тренировочных текстах.
4. В классе `ByteLevelBPETokenizer` реализуйте `bpe`, `encode` и `decode`.

При кодировании применяйте слияния с учётом выученных рангов. Базовый алфавит должен содержать все 256 значений байта; размер словаря учитывает также специальные токены.

In [ ]:
# Всякие полезности

WHITESPACE_SPLITTER = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


def bytes_to_unicode() -> dict[int, str]:
    """The original dictionary consists of 256 bytes and their corresponding Unicode characters.
    For example, chr(33) is '!'. However, not all bytes have a visually appealing representation,
    so such characters are skipped and replaced with the first available ones, i.e. shifted by 256.
    """
    initial_bytes = (
        list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    )
    initial_chars = [chr(it) for it in initial_bytes]
    n = 0
    for byte in range(2**8):
        if byte not in initial_bytes:
            initial_bytes.append(byte)
            initial_chars.append(chr(2**8 + n))
            n += 1
    return dict(sorted(zip(initial_bytes, initial_chars)))

In [ ]:
def merge(merge_pair: tuple[str, str], pair_frequences: Counter[tuple[str, str]], words_by_tokens: Counter[tuple[str]]):
    """Merges a given pair of tokens and update corresponding stats

    Args:
        merge_pair: The pair of tokens to be merged.
        pair_frequences: A counter tracking the frequency of token pairs in the dataset.
        words_by_tokens: A counter mapping tokenized words to their frequencies.

    Returns:
        Updated pair frequences and word tokenization w.r.t. to new token.
    """
    todo()


def train(data: list[str], vocab_size: int = 1024, special_tokens: list[str] = None):
    """Train BPE tokenizer on passed data

    Args:
        data: List of train documents
        vocab_size: Size of target vocabulary
        special_tokens: List of special tokens to add into vocabulary
    Returns:
        vocabulary: mapping from string token to id
        merges: list of merges, each one is tuple of string tokens
    """
    if vocab_size < 256:
        raise ValueError("Vocab size can't be less than 256")
    if special_tokens is None:
        special_tokens = []

    # 1. Initialize vocabulary (using inverse one during training)
    id2token = bytes_to_unicode()
    merges = []

    # 2. Load data
    words_by_tokens = Counter()
    for sample in tqdm(data, desc="Loading data"):
        # 2.1 Split into words
        words = WHITESPACE_SPLITTER.findall(sample.strip())
        for word in words:
            # 2.2 Tokenize with base vocabulary
            todo()

    # 3. Calculate statistic of token's pairs
    pair_frequences = Counter()
    todo()

    # 4. Build vocabulary
    pbar = trange(vocab_size, desc="Building vocabulary", initial=len(id2token) + len(special_tokens))
    while len(id2token) < vocab_size - len(special_tokens):
        if len(pair_frequences) == 0:
            print("Not enough data to fulfil vocabulary")
            break

        # 4.1 Find the most frequent pair and create new token
        top_pair = todo()
        new_token = todo()
        del pair_frequences[top_pair]

        # 4.2 Add to vocabulary
        if new_token in id2token.values():
            continue
        id2token[len(id2token)] = new_token
        merges.append(top_pair)

        # 4.3 Update stats and merge the top pair in all tokens
        pair_frequences, words_by_tokens = merge(top_pair, pair_frequences, words_by_tokens)

        pbar.update()
    pbar.close()

    # 5. Add special tokens
    for special_token in special_tokens:
        id2token[len(id2token)] = special_token

    return {v: k for k, v in id2token.items()}, merges

In [ ]:
# Обучаем токенизатор на тренировочных текстах
# Для нашей задачи хватит и небольшого словаря, но можете пробовать и большего размера обучить!


vocab, merges = train(dataset["train"]["text"], vocab_size=1024, special_tokens=["[EOS]"])

In [ ]:
# Посмотрим на случайные токены

random_tokens = [512, 614, 768, 888, 1022]
unicode_to_bytes = {v: k for k, v in bytes_to_unicode().items()}
for token_id in random_tokens:
    token = [k for k, v in vocab.items() if v == token_id][0]
    raw_bytes = bytes([unicode_to_bytes[it] for it in token])
    print(f"Token #{token_id}: '{raw_bytes.decode('utf-8', errors='replace')}'")

In [ ]:
class ByteLevelBPETokenizer:

    def __init__(self, vocab: dict[str, int], merges: list[tuple[str, str]], eos_token: str = "[EOS]"):
        """Byte-Level BPE Tokenizer

        Args:
            vocab: mapping from string token to id
            merges: list of merges in prioritized order
            eos_token: string representation of EOS token
        """
        super().__init__()
        if eos_token not in vocab:
            raise ValueError("There is no EOS token in vocab")
        self.byte_encoder = bytes_to_unicode()
        self.byte_decoder = {v: k for k, v in self.byte_encoder.items()}
        self.token2id = vocab
        self.id2token = {v: k for k, v in self.token2id.items()}
        self.eos_token = eos_token
        self.eos_token_id = self.token2id[eos_token]

        # Earlier merges have smaller numerical ranks and higher priority
        self.merges = merges
        self.bpe_ranks = {pair: i for i, pair in enumerate(merges)}

    @lru_cache
    def bpe(self, word: tuple[str]) -> tuple[str]:
        """Process word into tokenized representation.
        Word is a tuple of base tokens, i.e. bytes.

        Under the hood:
        1. Tracks the set of token pairs, bi-grams
        2. While possible, replaces the highest-ranking pair with its union

        Args:
            word: list of base string tokens
        Return:
            list of BPE tokens
        """
        todo()

    def encode(self, text: str, add_eos_token: bool = True) -> list[int]:
        """Convert string to list of token ids.

        Args:
            text: input string, may contain multiple words
            add_eos_token: whether to add eos token id at the end
        Return:
            list of ints, ids of tokenized text
        """
        words = WHITESPACE_SPLITTER.findall(text)
        todo()

    def decode(self, idx: list[int]) -> str:
        """Convert list of tokens' ids to text, opposite to encode method

        Args:
            idx: list of tokens' ids
        Return:
            string, decoded text
        """
        todo()

    def push_to_hub(self, repo_id, *, private=None, token=None):
        api = HfApi()
        repo_id = api.create_repo(repo_id=repo_id, token=token, private=private, exist_ok=True).repo_id

        # Push the files to the repo in a single commit
        with TemporaryDirectory() as tmp:
            save_directory = Path(tmp) / repo_id
            save_directory.mkdir(parents=True)
            with open(save_directory / "vocabulary.json", "w") as f_out:
                print(json.dumps(self.token2id, indent=2), file=f_out)
            with open(save_directory / "merges.json", "w") as f_out:
                print(json.dumps({"merges": self.merges}), file=f_out)

            return api.upload_folder(repo_id=repo_id, folder_path=save_directory, token=token)

    @classmethod
    def from_pretrained(cls, pretrained_model_name_or_path, *, token=None, **model_kwargs):
        if not os.path.isdir(pretrained_model_name_or_path):
            storage_folder = snapshot_download(repo_id=pretrained_model_name_or_path, token=token)
        else:
            storage_folder = pretrained_model_name_or_path
        storage_folder = Path(storage_folder)
        with open(storage_folder / "vocabulary.json", "r") as f_in:
            vocab = json.load(f_in)
        with open(storage_folder / "merges.json", "r") as f_in:
            merges = [tuple(it) for it in json.load(f_in)["merges"]]
        return cls(vocab, merges, **model_kwargs)

In [ ]:
# Инициализируем токенизатор


tokenizer = ByteLevelBPETokenizer(vocab, merges)

In [ ]:
# Смотрим на работу токенизатора

text = "Что было полгода назад? Помимо грандиозных событий, полгода назад были ещё семинары по линейной алгебре."
ids = tokenizer.encode(text)
print(ids)
reverse_text = [tokenizer.decode([it]) for it in ids]
print("|".join(reverse_text))
print(tokenizer.decode(ids))

In [ ]:
# Изучим длины токенизированных текстов на отложенной выборке

lens = []
for text in tqdm(dataset["test"]["text"]):
    ids = tokenizer.encode(text)
    lens.append(len(ids))

print(f"Average token len per sample: {sum(lens) / len(lens):.2f}")
print(f"Minimum and maximum lens are: {min(lens)} and {max(lens)}")

## Проверка кодирования и декодирования

После реализации методов выполните проверки ниже. `decode` должен удалять служебный EOS, если он присутствует, и восстанавливать исходный текст. Пустая строка, пробелы и Unicode не должны приводить к потере данных.

In [ ]:
samples = ["", "Привет, мир!", "  Пробелы по краям  ", "Строка\nс переносом", "Кот 🐈 и café"]
for sample in samples:
    token_ids = tokenizer.encode(sample, add_eos_token=False)
    assert all(isinstance(token_id, int) and token_id in tokenizer.id2token for token_id in token_ids)
    assert tokenizer.decode(token_ids) == sample

    token_ids_with_eos = tokenizer.encode(sample, add_eos_token=True)
    assert token_ids_with_eos[-1] == tokenizer.eos_token_id
    assert tokenizer.decode(token_ids_with_eos) == sample

print("Проверки восстановления текста пройдены.")

## Анализ результата

- Опишите несколько полученных токенов: соответствуют ли они словам, подсловам или отдельным байтам?
- Укажите среднюю, минимальную и максимальную длины последовательностей на отложенной выборке.
- Объясните, почему отдельный токен может не декодироваться в корректный символ UTF-8, хотя последовательность токенов восстанавливает исходный текст.
- При возможности сравните результаты для нескольких размеров словаря.

## Необязательно: публикация токенизатора

Этот раздел требует аккаунта Hugging Face и доступа к сети. Запускайте следующие ячейки только если хотите опубликовать готовый токенизатор. Укажите свой репозиторий; токен доступа вводится через форму авторизации, а не сохраняется в коде.

In [ ]:
interpreter_login()

In [ ]:
username = HfApi().whoami()["name"]
REPO_NAME = f"{username}/bpe-tokenizer-homework"
print(f"Репозиторий токенизатора: {REPO_NAME}")

In [ ]:
# Загружаем токенизатор на хаб

tokenizer.push_to_hub(REPO_NAME)

In [ ]:
restored_tokenizer = ByteLevelBPETokenizer.from_pretrained(REPO_NAME)
text = "Проверка сохранённого токенизатора."
assert restored_tokenizer.encode(text) == tokenizer.encode(text)
assert restored_tokenizer.decode(restored_tokenizer.encode(text)) == text